# 06 — Test SD3.5 inpaint-EDIT LoRA on PIPE golden set

Loads base + trained LoRA + `input_adapter.pt`, runs the full edit denoise loop
with hard-restore, and compares against base SD3.5 inpaint on the PIPE eval set
(real source/target pairs). Component metrics only; reference = real `target_img`.

## 1. Setup (install + restart, then verify)

In [ ]:
import subprocess, sys, os
from pathlib import Path
REPO = Path('/kaggle/working/VIN')
if not REPO.exists():
    subprocess.run(['git','clone','https://github.com/BDT-17/VIN.git',str(REPO)], check=True)
else:
    subprocess.run(['git','-C',str(REPO),'fetch','origin'], check=True)
    subprocess.run(['git','-C',str(REPO),'reset','--hard','origin/main'], check=True)
sys.path.insert(0, str(REPO))
print('repo at', subprocess.run(['git','-C',str(REPO),'rev-parse','--short','HEAD'],capture_output=True,text=True).stdout.strip())
!pip install -q --force-reinstall --no-deps 'transformers==4.46.3' 'tokenizers==0.20.3' 'huggingface_hub==0.25.2'
!pip install -q 'diffusers==0.31.0' 'accelerate==0.34.2' 'peft==0.13.2' 'datasets>=2.20' 'safetensors>=0.4.3' 'sentencepiece' 'protobuf' 'pillow>=10' numpy
print('Installed. Restarting...'); os._exit(0)

In [ ]:
import os, sys
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'
sys.path.insert(0, '/kaggle/working/VIN')
import transformers, diffusers, torch
assert transformers.__version__ == '4.46.3'
from transformers.utils import FLAX_WEIGHTS_NAME
print('ok', diffusers.__version__, torch.cuda.get_device_name(0))

## 2. SD3.5 access + build PIPE eval set (if needed)

In [ ]:
from pathlib import Path
_local = Path('/kaggle/input/stable-diffusion-3-5-medium')
HF_TOKEN = None
if _local.exists():
    SD35_MODEL = str(_local)
else:
    SD35_MODEL = 'stabilityai/stable-diffusion-3.5-medium'
    try:
        from kaggle_secrets import UserSecretsClient; HF_TOKEN = UserSecretsClient().get_secret('HF_TOKEN')
    except Exception:
        import os; HF_TOKEN = os.environ.get('HF_TOKEN')
    assert HF_TOKEN, 'Need HF_TOKEN or local SD3.5 mount'
    from huggingface_hub import login; login(token=HF_TOKEN)

from LoRA.data.build_eval_cases_pipe import run_build_pipe_eval
import json
WORK = Path('/kaggle/working/vin_lora')
EVAL = WORK/'eval'/'pipe_eval_v1'
if not (EVAL/'cases.jsonl').exists():
    run_build_pipe_eval(WORK, eval_set='pipe_eval_v1', split='test', person_only=True, limit=40)
cases = [json.loads(l) for l in (EVAL/'cases.jsonl').read_text().splitlines() if l.strip()]
print(len(cases), 'eval cases')

## 3. Load the trained edit adapter (LoRA + input_adapter.pt)

In [ ]:
from LoRA.inference.sd35_edit_runner import load_edit_runner_from_run
RUN_DIR = WORK/'models'/'vinped_sd35m_edit_v1'/'run_001'   # adjust to your run
runner, prov = load_edit_runner_from_run(RUN_DIR, base_model_id=SD35_MODEL, hf_token=HF_TOKEN)
assert prov.get('requires_input_adapter'), 'provenance missing input-adapter flag'
print('loaded edit adapter from', RUN_DIR)

## 4. Run edit LoRA on the eval set + metrics vs real target
(Background is hard-restored, so outside_mask stays ~0; the signal is in-mask.)

In [ ]:
from PIL import Image
from LoRA.inference.inpaint_metrics import compute_case_metrics
RUN = WORK/'runs'/'edit_pipe_eval_v1'
(RUN/'lora'/'images').mkdir(parents=True, exist_ok=True)

# One-time: encode ALL prompts (text encoders on GPU), then drop them so the
# denoise loop has room on the T4. MUST run before any edit().
def case_prompt(c):
    return 'a photo of <vin_ped> pedestrian, ' + (c['prompt_fields'].get('instruction','') or 'a person')
runner.precompute_embeds([case_prompt(c) for c in cases])

rows = []
for c in cases:
    src = Image.open(EVAL/c['image_path']); msk = Image.open(EVAL/c['mask_path'])
    prompt = case_prompt(c)
    for seed in [42, 43, 44]:
        out = runner.edit(src, msk, prompt, seed=seed, num_inference_steps=30)
        p = RUN/'lora'/'images'/f"{c['case_id']}_s{seed}.png"; out.save(p)
        m = compute_case_metrics(EVAL/c['reference_path'], p, EVAL/c['mask_path'],
                                 c['expected_bbox_xyxy'], detector=None)
        m.update({'case_id': c['case_id'], 'seed': seed}); rows.append(m)
print('generated', len(rows), 'edit results ->', RUN)

## 5. Contact sheet (manual check): source | mask | edit result

In [ ]:
from PIL import Image
import csv
with open(RUN/'edit_metrics.csv','w',newline='') as f:
    w = csv.DictWriter(f, fieldnames=sorted({k for r in rows for k in r})); w.writeheader(); w.writerows(rows)
# 1 contact row per case (seed 42)
thumbs = []
for c in cases[:8]:
    src = Image.open(EVAL/c['image_path']).resize((256,256))
    msk = Image.open(EVAL/c['mask_path']).convert('RGB').resize((256,256))
    res = Image.open(RUN/'lora'/'images'/f"{c['case_id']}_s42.png").resize((256,256))
    strip = Image.new('RGB',(768,256)); strip.paste(src,(0,0)); strip.paste(msk,(256,0)); strip.paste(res,(512,0))
    thumbs.append(strip)
sheet = Image.new('RGB',(768,256*len(thumbs)))
for i,s in enumerate(thumbs): sheet.paste(s,(0,256*i))
sheet.save(RUN/'contact_sheet.png'); print('contact sheet ->', RUN/'contact_sheet.png')
from IPython.display import Image as IPImage, display; display(IPImage(str(RUN/'contact_sheet.png')))